In [ ]:
%pip install -Uq langchain langchain-openai python-dotenv langchain-cohere langchain-community pypdf beautifulsoup4

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv() # carrega a API da I.A do .env

model = init_chat_model("openai:gpt-4o-mini")

In [ ]:
from langchain_core.documents import Document
doc = Document(
    page_content="A LangChain v1 cria agents com create_agent e tools com @tool.",
    metadata={"fonte": "notas.md", "secao": "intro"},
)
print(doc.page_content)
print(doc.metadata["fonte"])

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("dados/manual.txt", encoding="utf-8")
docs = loader.load()        # -> list[Document] (1 Document com o arquivo inteiro)
print(len(docs), docs[0].metadata)      # metadata inclui {"source": "dados/manual.txt"}

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("dados/regras.pdf") # pode ser URL também
# ex de url: https://www2.senado.leg.br/bdsf/bitstream/handle/id/518231/CF88_Livro_EC91_2016.pdf
docs = loader.load()
print(f"{len(docs)} páginas carregadas")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://www2.senado.leg.br/bdsf/bitstream/handle/id/518231/CF88_Livro_EC91_2016.pdf") # pode ser URL também
# ex de url: https://www2.senado.leg.br/bdsf/bitstream/handle/id/518231/CF88_Livro_EC91_2016.pdf
docs = loader.load()
print(f"{len(docs)} páginas carregadas")

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    "https://pt.wikipedia.org/wiki/Estatuto_da_Crian%C3%A7a_e_do_Adolescente",
    bs_get_text_kwargs={"separator": " ", "strip": True},
    # o WebBaseLoader usa a biblioteca BeautifulSoup por baixo
    # e você pode passar opções específicas
)
docs = loader.load()
print(len(docs[0].page_content))

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Carrega TODOS os .md de uma pasta (e subpastas), usando o TextLoader em cada um.
loader = DirectoryLoader(
    "dados/",
    glob="**/*.md",                 # padrão dos arquivos a incluir
    loader_cls=TextLoader,          # qual loader usar por arquivo
    loader_kwargs={"encoding": "utf-8"},
)
docs = loader.load()
print(f"{len(docs)} arquivos carregados")

In [ ]:
%pip install -U langchain-unstructured unstructured

from langchain_unstructured import UnstructuredLoader

# Um arquivo - o tipo é detectado automaticamente
loader = UnstructuredLoader("dados/relatorio.pdf")
docs = loader.load()

# Vários arquivos de tipos DIFERENTES de uma vez
loader = UnstructuredLoader(["relatorio.pdf", "notas.docx", "pagina.html"])
docs = loader.load()

In [ ]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader("dados/", glob="**/*")     # todos os arquivos, tipos variados
docs = loader.load()

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_unstructured import UnstructuredLoader

LOADERS = {".pdf": PyPDFLoader, ".docx": Docx2txtLoader, "txt": TextLoader, ".md": TextLoader}

def carregar_qualquer(caminho: str):
    ext = Path(caminho).suffix.lower()
    loader_cls = LOADERS.get(ext, UnstructuredLoader) #fallback: unstructured
    return loader_cls(caminho).load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # tamanho-alvo de cada chunk (em caracteres)
    chunk_overlap=200,      # sobreposição entre chunks vizinhos
    add_start_index=True,   # guarda em metadata a posição original do chunk
)

chunks = splitter.split_documents(docs)     # docs = list[Document] (vindo do loader)
print(f"{len(docs)} documentos viraram {len(chunks)} chunks")

In [ ]:
# Divisão por TOKENS (precisa: pip install tiktoken)
splitter_tokens = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,      # agora em TOKENS, não caracteres
    chunk_overlap=50,
)

# Divisão por TÍTULOS de Markdown (mantém a seção no metadata)
from langchain_text_splitters import MarkdownHeaderTextSplitter

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")],
)
secoes = md_splitter.split_text(texto_markdown)     # cada chunk carrega o título em metadata